<a href="https://colab.research.google.com/github/PeterMaZep/SyntheticMcWikiGenerator/blob/main/Meta_Synthetic_Data_Llama3_2_(3B).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!

Custom version of Synthetic Data Generator for Minecraft Wiki!

### News

March update 2026 in progress!

### Installation

In [ ]:
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm synthetic-data-kit==0.0.3
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    !uv pip install synthetic-data-kit==0.0.3
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

In [ ]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

### Unsloth

## Primary Goal
Our goal is to make Llama 3.2 3B understand the "Byte Latent Transformer: Patches Scale Better Than Tokens" [research paper](https://ai.meta.com/research/publications/byte-latent-transformer-patches-scale-better-than-tokens/) that was published in December 2024.

We'll use https://github.com/meta-llama/synthetic-data-kit to generate question and answer pairs **fully locally** which will be used for finetuning Llama 3.2 3B!

In [ ]:
from unsloth.dataprep import SyntheticDataKit

generator = SyntheticDataKit.from_pretrained(
    # Choose any model from https://huggingface.co/unsloth
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = 2048, # Longer sequence lengths will be slower!
)

## Generate QA Pairs + Auto clean data
We now use synthetic data kit for question answer pair generation:

In [ ]:
generator.prepare_qa_generation(
    output_folder = "data", # Output location of synthetic data
    temperature = 0.7, # Higher temp makes more diverse datasets
    top_p = 0.95,
    overlap = 64, # Overlap portion during chunking
    max_generation_tokens = 512, # Can increase for longer QA pairs
)

Check if it succeeded:

In [ ]:
!synthetic-data-kit system-check

## Document Parsing (PDF, CSV, HTML etc.)
Now, let's take the Byte Latent Transformer: Patches Scale Better Than Tokens research paper found at https://arxiv.org/abs/2412.09871 and covert it to Q&A pairs in order to finetune Llama 3.2!

## Loading previous data


In [ ]:
from datasets import load_dataset
import json, os, glob

HF_REPO = "PWindows/minecraft-wiki-qa"
from google.colab import userdata
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("✓ Token loaded from Colab secrets")
except:
    import getpass
    HF_TOKEN = getpass.getpass("Enter HuggingFace token: ")

# Ask user whether to resume or restart
choice = input("\nResume from previous session or restart? (resume/restart): ").strip().lower()

if choice == "restart":
    completed_pages = set()
    print("✓ Starting fresh")
else:
    try:
        existing = load_dataset(HF_REPO, token=HF_TOKEN)
        completed_pages = set(existing["train"]["source"])
        print(f"✓ Resuming — {len(completed_pages)} pages already done")
    except:
        completed_pages = set()
        print("✓ No existing dataset found, starting fresh")

## Fetching wiki


In [ ]:
import requests, time

WIKI_API = "https://minecraft.wiki/api.php"

def get_all_titles():
    titles = []
    params = {"action":"query","list":"allpages","aplimit":"500","format":"json","apnamespace":"0"}
    while True:
        r = requests.get(WIKI_API, params=params, timeout=30).json()
        titles.extend([p["title"] for p in r["query"]["allpages"]])
        if "continue" not in r: break
        params["apcontinue"] = r["continue"]["apcontinue"]
        time.sleep(0.5)
    return titles

def get_page_text(title):
    params = {"action":"query","titles":title,"prop":"extracts","explaintext":True,"format":"json"}
    r = requests.get(WIKI_API, params=params, timeout=30).json()
    page = next(iter(r["query"]["pages"].values()))
    text = page.get("extract", "")
    return text[:3000] if len(text) > 200 else None

titles = get_all_titles()
print(f"✓ Found {len(titles)} pages")

# QA Generation

In [ ]:
import time, json, glob, os
from datasets import Dataset

os.makedirs("data/input", exist_ok=True)
BATCH_SIZE = 50
batch = []

for i, title in enumerate(titles):
    if title in completed_pages:
        continue

    print(f"[{i}/{len(titles)}] {title}")

    text = get_page_text(title)
    if not text:
        print("  Skipping (no content)")
        completed_pages.add(title)
        continue

    # Save to temp file
    with open("data/input/page.txt", "w") as f:
        f.write(text)

    try:
        # Chunk and generate
        chunks = generator.chunk_data("data/input/page.txt")
        for chunk in chunks[:1]:
            !synthetic-data-kit \
                -c synthetic_data_kit_config.yaml \
                create {chunk} \
                --num-pairs 7 \
                --type "qa"
            time.sleep(2)

        # Load generated pairs
        for fn in glob.glob("data/generated/page_*_qa_pairs.json"):
            with open(fn) as f:
                pairs = json.load(f)
            for p in pairs:
                batch.append({
                    "question": p["question"],
                    "answer": p["answer"],
                    "source": title
                })
            os.remove(fn)  # cleanup

    except Exception as e:
        print(f"  Failed: {e}")
        continue

    completed_pages.add(title)

    if len(batch) >= BATCH_SIZE:
        try:
            existing = load_dataset(HF_REPO, token=HF_TOKEN)
            combined = existing["train"].to_list() + batch
        except:
            combined = batch
        Dataset.from_list(combined).push_to_hub(HF_REPO, token=HF_TOKEN)
        print(f"✓ Pushed {len(batch)} pairs")
        batch.clear()

# Final push
if batch:
    try:
        existing = load_dataset(HF_REPO, token=HF_TOKEN)
        combined = existing["train"].to_list() + batch
    except:
        combined = batch
    Dataset.from_list(combined).push_to_hub(HF_REPO, token=HF_TOKEN)
    print(f"✓ Final push: {len(batch)} pairs")

# Cleaning up


In [ ]:
generator.cleanup()

And we're done!
